# Análisis Exploratorio de Datos: World Happiness Report con Seaborn

Este notebook realiza un análisis exploratorio de datos (EDA) utilizando el dataset del **World Happiness Report** de 2015 a 2021, enfocándose en el uso de **Seaborn** para visualizaciones estadísticas avanzadas.

## ¿Por qué Seaborn?

Seaborn está construido sobre Matplotlib y ofrece ventajas clave para EDA:
- **Menos código**: Las visualizaciones complejas (barras agrupadas, facetas, heatmaps) se crean con una sola llamada.
- **Estadísticas integradas**: Calcula y muestra intervalos de confianza, KDE y regresiones automáticamente.
- **Estilos modernos**: Temas visuales atractivos sin configuración adicional.
- **Integración con DataFrames**: Acepta directamente columnas de pandas como parámetros `x`, `y`, `hue`.

## Contexto del Dataset

El World Happiness Report clasifica a los países según su nivel de felicidad percibida, combinando encuestas (escala Cantril) con datos económicos y sociales. Las métricas principales son:

| Variable | Descripción |
|---|---|
| `Happiness Score` | Puntuación global de bienestar (0–10) |
| `Economy (GDP per Capita)` | PIB per cápita normalizado |
| `Family (Social Support)` | Red de apoyo social percibida |
| `Health (Life Expectancy)` | Esperanza de vida saludable |
| `Freedom` | Libertad para tomar decisiones vitales |
| `Trust (Government Corruption)` | Percepción de honestidad gubernamental |
| `Generosity` | Donaciones como % del PIB |
| `Region` | Agrupación geográfico-cultural del país |

## Estructura del Notebook

| Ejercicio | Tipo de Gráfico | Objetivo |
|---|---|---|
| 1 | `lineplot` | Evolución temporal de la felicidad |
| 2 | `barplot` | Comparación del PIB per cápita |
| 3 | `displot` | Distribución del PIB por año (facetas) |
| 4 | `relplot` | Relación GDP vs. soporte social por nivel de felicidad |
| 5 | `barplot` horizontal | Confianza en gobierno por región |
| 6 | `heatmap` | Matriz de correlación entre variables |

In [ ]:
%pip install pandas matplotlib seaborn --quiet

In [ ]:
## ─── Configuración global ────────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo visual de Seaborn: 'whitegrid' añade una cuadrícula blanca sobre fondo
# gris claro, mejorando la legibilidad sin saturar el gráfico.
sns.set_theme(style="whitegrid", palette="muted")

# Tamaño de figura por defecto para todos los gráficos del notebook
plt.rcParams["figure.figsize"] = (10, 5)

## ─── Carga del dataset ───────────────────────────────────────────────────────
# Se carga una sola vez y se reutiliza en todos los ejercicios.
# El archivo CSV combina los informes anuales (2015–2021) con la columna 'Year'.
report = pd.read_csv('world-happiness-report.csv')

# Vista rápida: dimensiones y primeras filas
print(f"Filas: {report.shape[0]} | Columnas: {report.shape[1]}")
print(f"Años disponibles: {sorted(report['Year'].unique())}")
report.head()

## Ejercicio 1: Evolución de la Puntuación de Felicidad (`sns.lineplot`)

### Objetivo
Visualizar cómo ha evolucionado la felicidad en España, Francia y Portugal entre 2015 y 2021, permitiendo comparar tendencias a lo largo del tiempo.

### Función principal: `sns.lineplot()`
```python
sns.lineplot(data=df, x="columna_x", y="columna_y", hue="columna_color")
```
- `hue`: Crea una línea diferente por cada valor único de la columna indicada, asignando colores automáticamente y generando la leyenda.
- Sin `hue`, dibujaría una sola línea con el promedio global.

### Ventaja frente a Matplotlib puro
En Matplotlib habría que filtrar cada país por separado y llamar a `plt.plot()` tres veces. Seaborn hace todo eso con un único `lineplot`.

### Qué buscar en el resultado
- ¿Hay tendencia al alza o a la baja en algún país?
- ¿Los tres países muestran patrones similares o divergen?
- Las caídas pronunciadas suelen coincidir con crisis económicas o sanitarias (ej. 2020).

In [ ]:
# ── Filtrar los tres países de interés ───────────────────────────────────────
# Se utiliza el operador '|' (OR) para seleccionar filas de múltiples países.
# Alternativa equivalente más compacta: report[report['Country'].isin([...])]
paises = ['Spain', 'France', 'Portugal']
report_filtered = report[report['Country'].isin(paises)]

# ── Gráfico de líneas ─────────────────────────────────────────────────────────
# hue='Country' → una línea por país, con color y leyenda automáticos
# markers=True  → añade un marcador en cada punto de dato para claridad
sns.lineplot(
    data=report_filtered,
    x="Year",
    y="Happiness Score",
    hue="Country",
    markers=True,
    dashes=False       # líneas continuas en lugar de discontinuas
)

plt.title("Evolución de la Puntuación de Felicidad: España, Francia y Portugal")
plt.xlabel("Año")
plt.ylabel("Puntuación de Felicidad (0–10)")
plt.legend(title="País")
plt.tight_layout()
plt.show()

## Ejercicio 2: Evolución del PIB per Cápita (`sns.barplot`)

### Objetivo
Comparar el PIB per cápita de España, Francia y Portugal para cada año disponible, usando barras agrupadas.

### Función principal: `sns.barplot()`
```python
sns.barplot(data=df, x="año", y="valor", hue="pais")
```
- Con `hue`, Seaborn agrupa las barras automáticamente y ajusta el ancho para que quepan en el mismo eje.
- **Barras de error**: Por defecto muestra el intervalo de confianza del 95% (IC95). Si hay un único valor por combinación año-país, la barra de error no aparece.

### Diferencia clave con `sns.lineplot`
- `lineplot` conecta puntos → ideal para **tendencias continuas**.
- `barplot` muestra magnitudes discretas → ideal para **comparar categorías** (años, países).

### Qué buscar en el resultado
- ¿Cuál de los tres países tiene mayor PIB per cápita de forma consistente?
- ¿Se observa crecimiento sostenido o hay años de caída?

In [ ]:
# ── Reutilizamos el filtro del ejercicio anterior ────────────────────────────
# 'report_filtered' ya contiene los datos de España, Francia y Portugal.
# Nota: si ejecutas esta celda sin haber ejecutado la anterior, define:
# paises = ['Spain', 'France', 'Portugal']
# report_filtered = report[report['Country'].isin(paises)]

# ── Barras agrupadas por país y año ───────────────────────────────────────────
# hue='Country' → una barra por país dentro de cada grupo de año
# errorbar=None → desactiva las barras de error para simplificar la lectura
#                 (hay un único valor por combinación país-año)
sns.barplot(
    data=report_filtered,
    x='Year',
    y='Economy (GDP per Capita)',
    hue='Country',
    errorbar=None      # equivalente a ci=None en versiones antiguas de Seaborn
)

plt.title("Evolución del PIB per Cápita: España, Francia y Portugal")
plt.xlabel("Año")
plt.ylabel("PIB per Cápita (valor normalizado)")
plt.legend(title="País")
plt.tight_layout()
plt.show()

## Ejercicio 3: Distribución del PIB por Año (`sns.displot` con facetas)

### Objetivo
Examinar cómo varía la distribución global del PIB per cápita en cada año, creando un panel de histogramas.

### Función principal: `sns.displot()`
```python
sns.displot(data=df, x="variable", col="faceta", col_wrap=N)
```
- `col="Year"`: crea un subplot separado por cada valor único de `Year`.
- `col_wrap=2`: limita el número de columnas por fila; útil para no hacer el panel demasiado ancho.
- `kind='hist'` (defecto): histograma. También acepta `'kde'` (curva de densidad) o `'ecdf'`.

### Concepto clave: Facetas (FacetGrid)
Las facetas permiten ver **la misma visualización para distintos subgrupos** sin necesidad de bucles ni `plt.subplot()` manual. Seaborn internamente usa un `FacetGrid`.

### Qué buscar en el resultado
- ¿La distribución se desplaza hacia la derecha con los años? (indicaría crecimiento global del PIB)
- ¿Aparecen dos "jorobas" (distribución bimodal)? Esto señalaría desigualdad entre países ricos y pobres.
- ¿Hay outliers persistentes en el extremo derecho? (países con PIB muy alto como Luxemburgo o Qatar)

In [ ]:
# ── Panel de histogramas: una faceta por año ─────────────────────────────────
# displot devuelve un objeto FacetGrid, no un Axes de Matplotlib.
# Por eso la personalización se hace con métodos del propio objeto 'g'.
g = sns.displot(
    data=report,
    x="Economy (GDP per Capita)",
    col="Year",          # una columna por año
    col_wrap=2,          # máximo 2 columnas por fila
    bins=20,             # número de barras del histograma
    kde=True,            # superpone curva de densidad (KDE) para ver la forma
    height=3.5,          # altura de cada faceta en pulgadas
    aspect=1.4           # proporción ancho/alto de cada faceta
)

# Personalización del FacetGrid (no se usa plt.title ni plt.xlabel)
g.set_titles("Año: {col_name}")           # título de cada panel
g.set_axis_labels("PIB per Cápita", "Frecuencia")
g.figure.suptitle(
    "Distribución del PIB per Cápita por Año",
    y=1.02, fontsize=13  # suptitle queda fuera del grid → y>1 evita solapamiento
)
plt.tight_layout()
plt.show()

## Ejercicio 4: Relación entre PIB y Soporte Social (`sns.relplot`)

### Objetivo
Explorar la relación entre el PIB per cápita y el soporte social, coloreando los puntos según el nivel de felicidad del país.

### Función principal: `sns.relplot()`
```python
sns.relplot(data=df, x="var_x", y="var_y", hue="categoria")
```
- `relplot` es el scatter plot "de nivel figura" de Seaborn (equivale a `scatterplot` dentro de un `FacetGrid`).
- Con `hue` categórico, asigna colores discretos por clase.
- También acepta `size` y `style` para codificar hasta 4 variables en un mismo gráfico.

### Categorización con `pd.qcut()` vs `pd.cut()`

| Función | Divide por... | Cuándo usarla |
|---|---|---|
| `pd.cut(bins=[0,4,6,10])` | Intervalos de igual anchura | Cuando los límites tienen sentido fijo (ej. notas 0–5–10) |
| `pd.qcut(q=3)` | Cuantiles (misma cantidad de datos) | Cuando importa el ranking relativo, no el valor absoluto |

Usamos `pd.qcut` porque la escala de felicidad no tiene un umbral "natural" para LOW/MED/HIGH.

### Qué buscar en el resultado
- ¿Los países HIGH están concentrados en la esquina superior derecha (más PIB y más soporte)?
- ¿Existe relación lineal o se satura a partir de cierto nivel de PIB?

In [ ]:
# ── Categorizar el Happiness Score en 3 niveles por cuantiles ────────────────
# pd.qcut divide los datos en 'q' grupos de igual frecuencia (cuantiles).
# Resultado: ~33% de países en cada categoría LOW, MED, HIGH.
labels = ['LOW', 'MED', 'HIGH']

# Trabajamos sobre una copia para no modificar el DataFrame original
report_cat = report.copy()
report_cat['Range'] = pd.qcut(report_cat['Happiness Score'], q=3, labels=labels)

# ── Scatter plot coloreado por nivel de felicidad ─────────────────────────────
# alpha: transparencia para ver puntos solapados (0=transparente, 1=sólido)
# palette: 'RdYlGn' → rojo (LOW), amarillo (MED), verde (HIGH)
g = sns.relplot(
    data=report_cat,
    x="Economy (GDP per Capita)",
    y="Family (Social Support)",
    hue="Range",
    hue_order=labels,          # orden explícito LOW → MED → HIGH
    palette="RdYlGn",
    alpha=0.7,
    height=5,
    aspect=1.6
)

g.set_axis_labels("PIB per Cápita", "Soporte Social (Family)")
g.figure.suptitle(
    "Relación PIB per Cápita vs. Soporte Social según Nivel de Felicidad",
    y=1.02
)
plt.tight_layout()
plt.show()

## Ejercicio 5: Confianza en el Gobierno por Región (`sns.barplot` horizontal)

### Objetivo
Comparar la media de confianza en el gobierno entre las distintas regiones del mundo, ordenadas de menor a mayor.

### Técnica: barras horizontales
Se invierten los ejes (`x=valor_numérico`, `y=categoría_texto`) para que los nombres de región sean legibles sin rotación.

### Preparación de datos con `groupby`
```python
df.groupby('Region')['columna'].mean()
```
1. `groupby('Region')`: agrupa todas las filas por región.
2. `['columna'].mean()`: calcula la media de esa columna dentro de cada grupo.
3. `sort_values()`: ordena el resultado de menor a mayor para que el gráfico sea más informativo.

### Paleta `coolwarm`
- Colores fríos (azul) para valores bajos → baja confianza.
- Colores cálidos (rojo) para valores altos → alta confianza.
- Es una paleta **divergente**, ideal para destacar extremos respecto a una media central.

### Qué buscar en el resultado
- ¿Existe una brecha grande entre la región con más y menos confianza?
- Las regiones nórdicas y de Oceanía suelen liderar; Latinoamérica y África subsahariana tienden a la parte baja.

In [ ]:
# ── Calcular la confianza media por región ───────────────────────────────────
# groupby + mean + dropna: ignora los NaN en el cálculo de la media.
# reset_index() convierte el índice 'Region' en columna para pasarlo a Seaborn.
trust = (
    report.groupby('Region')['Trust (Government Corruption)']
    .mean()
    .dropna()
    .sort_values()            # ascendente: la región menos confiable queda arriba
    .reset_index()            # convierte el índice 'Region' en columna
)

# ── Barras horizontales ordenadas ─────────────────────────────────────────────
# x → valor numérico; y → categoría textual (barras horizontales automáticas)
# palette='coolwarm' codifica visualmente el valor: azul=bajo, rojo=alto
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=trust,
    x='Trust (Government Corruption)',
    y='Region',
    palette='coolwarm',
    ax=ax
)

ax.set_title("Confianza en el Gobierno por Región (Media 2015–2021)")
ax.set_xlabel("Índice de Confianza (media)")
ax.set_ylabel("")  # el eje Y ya es autoexplicativo con los nombres de región
plt.tight_layout()
plt.show()

## Ejercicio 6: Mapa de Calor de Correlaciones (`sns.heatmap`)

### Objetivo
Identificar qué variables están más fuertemente relacionadas con la felicidad y entre sí.

### Función principal: `sns.heatmap()`
```python
sns.heatmap(data=matriz_correlacion, annot=True, fmt=".2f", cmap='coolwarm')
```
- `data`: espera una **matriz cuadrada** (filas y columnas son las mismas variables).
- `annot=True`: muestra el valor numérico dentro de cada celda.
- `fmt=".2f"`: formatea los números a 2 decimales.
- `cmap='coolwarm'`: paleta divergente (azul para correlaciones negativas, rojo para positivas).
- `linewidths`: añade líneas entre celdas para facilitar la lectura.

### Correlación de Pearson
El método `.corr()` calcula la correlación de **Pearson** por defecto:
- Valor en `[-1, 1]`
- `+1`: perfectamente correlacionadas en la misma dirección
- `0`: sin relación lineal
- `-1`: perfectamente correlacionadas en sentido inverso

> **Nota**: La diagonal principal siempre vale 1 (una variable correlaciona perfectamente consigo misma).

### Qué buscar en el resultado
- Las variables con mayor correlación con `Happiness Score` son los predictores más importantes.
- Correlaciones altas entre predictores (multicolinealidad) pueden afectar a modelos de regresión.
- ¿Tiene `Generosity` la correlación más baja con la felicidad?

In [ ]:
# ── Seleccionar solo las variables numéricas de interés ──────────────────────
# Excluimos 'Year' porque es un identificador temporal, no una variable explicativa.
columnas_numericas = [
    "Happiness Score",
    "Economy (GDP per Capita)",
    "Health (Life Expectancy)",
    "Family (Social Support)",
    "Freedom",
    "Trust (Government Corruption)",
    "Generosity"
]

# ── Calcular la matriz de correlación de Pearson ─────────────────────────────
# .corr() ignora automáticamente las filas con NaN en pares de columnas.
correlacion = report[columnas_numericas].corr()

# ── Heatmap anotado ───────────────────────────────────────────────────────────
# mask: oculta el triángulo superior para evitar redundancia (la matriz es simétrica)
import numpy as np
mask = np.triu(np.ones_like(correlacion, dtype=bool))   # True = celda oculta

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    data=correlacion,
    mask=mask,             # muestra solo el triángulo inferior
    cmap='coolwarm',
    annot=True,            # muestra el valor numérico en cada celda
    fmt=".2f",             # 2 decimales
    linewidths=0.5,        # separación visual entre celdas
    vmin=-1, vmax=1,       # rango fijo para que los colores sean comparables
    square=True,           # celdas cuadradas
    ax=ax
)

ax.set_title("Mapa de Calor de Correlaciones (Pearson) — World Happiness Report", pad=12)
plt.tight_layout()
plt.show()

# ── Resumen: variables más correlacionadas con la felicidad ──────────────────
print("\nCorrelación con 'Happiness Score' (de mayor a menor):")
print(correlacion["Happiness Score"].drop("Happiness Score").sort_values(ascending=False).to_string())

## Conclusión del Análisis con Seaborn

### Resumen de hallazgos

| Ejercicio | Hallazgo principal |
|---|---|
| 1 – Evolución temporal | Francia lidera consistentemente; los tres países muestran estabilidad relativa |
| 2 – PIB per cápita | Francia > España > Portugal en toda la serie temporal |
| 3 – Distribución global del PIB | Distribución sesgada a la derecha; la mayoría de países tiene PIB bajo–medio |
| 4 – PIB vs. Soporte Social | Clara separación entre categorías: HIGH concentra PIB y soporte social altos |
| 5 – Confianza gubernamental | Alta variabilidad regional; diferencia de ~5x entre la región más y menos confiante |
| 6 – Correlaciones | `Economy` y `Health` son los predictores más fuertes de felicidad; `Generosity` el más débil |

### Ventajas demostradas de Seaborn
- **Código más limpio**: cada visualización requirió menos líneas que su equivalente en Matplotlib.
- **Estadísticas integradas**: KDE, intervalos de confianza y matrices de correlación sin librerías adicionales.
- **Semántica declarativa**: se describe *qué* mostrar (`hue`, `col`, `palette`), no *cómo* dibujarlo.

### Próximos pasos sugeridos
1. **`sns.pairplot()`**: visualizar todas las correlaciones par a par en un solo comando.
2. **`sns.lmplot()`**: añadir líneas de regresión al scatter para cuantificar relaciones.
3. **`sns.boxplot()` / `sns.violinplot()`**: comparar distribuciones de felicidad por región.
4. Integrar estos insights en un modelo predictivo (regresión lineal o árbol de decisión).